# M4 — conditional RBF-kernel copula

This self-contained notebook explains the fifth principal dependence family. Its scientific restriction is geometric: an RBF kernel represents nonnegative similarity in a learned space. Its execution status, partitions, and evaluation are otherwise equal to M2 and M3. See [Chapter 4](../../docs/scientific/04_dependence_models.md).

## Notation and kernel hypothesis

$i$ indexes origin $t^{(i)}$, $g$ indexes $\mathcal E_g=\{k_1,\ldots,k_{K_g}\}$, and $\tau$ is one-based lead. $Y$ and $y$ denote future random variables and observations, $F$ is the fixed marginal, and $A_{g,\tau}^{(i)}=\sum_kY_{k,\tau}^{(i)}$. Define $V_{k,\tau}^{(i)}\in\{0,1\}$ and $V_{g,\tau}^{(i)}=\prod_kV_{k,\tau}^{(i)}$; M4 is defined only when this product is one. A shared network embeds each origin-measurable vector as $h_k=g_\theta(v_{k,\tau}^{(i)})\in\mathbb R^d$. M4 forms
$$K_{ab}=\exp[-\|h_a-h_b\|_2^2/(2\ell^2)]+\nu\,\mathbf1(a=b),$$
then normalizes $K$ to a correlation $R_{g,\tau}^{(i)}\in\mathbb R^{K_g\times K_g}$. $\ell>0$ is length scale and $\nu>0$ is a nugget. The RBF Gram matrix is positive semidefinite and permutation equivariant. Its off-diagonal elements are nonnegative, so it cannot express negative residual rank association.

## Meaning of the length scale

At embedded distance $d_{ab}$, correlation before nugget normalization is $\exp[-d_{ab}^2/(2\ell^2)]$. A small $\ell$ permits rapid decay; a large $\ell$ keeps distant embeddings similar. The embedding distance is learned and is not automatically a geographical distance.

In [ ]:
from pathlib import Path
import sys, numpy as np, matplotlib.pyplot as plt
PROJECT_ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'pyproject.toml').is_file())
if str(PROJECT_ROOT/'src') not in sys.path: sys.path.insert(0,str(PROJECT_ROOT/'src'))
from simcast.config import load_base_config, load_method_config, resolve_run_config
from simcast.experiments import locate_compatible_cache
from simcast.fm.cache import load_pit_library
from simcast.cli.train_dependence import train_from_config
from simcast.cli.evaluate import evaluate_from_config
BASE_FILE='configs/bases/liander2024/transformer.yaml'
METHOD_FILE='configs/methods/m4_conditional_kernel.yaml'
BASE_OVERRIDES=(); METHOD_OVERRIDES=()
TRAIN_IF_MISSING=False; RUN_EVALUATION=False
RUN_DIR=PROJECT_ROOT/'runs'/'notebook_walkthrough'/'m4_conditional_kernel'
base=load_base_config(PROJECT_ROOT/BASE_FILE,overrides=BASE_OVERRIDES)
method=load_method_config(PROJECT_ROOT/METHOD_FILE,overrides=METHOD_OVERRIDES)
config=resolve_run_config(base,method)
cache_dir=locate_compatible_cache(base)
library=load_pit_library(cache_dir,access='training'); ds=library.dataset
entity_ids=[str(x) for x in ds.entity_id.values]; K_g=len(entity_ids)
s=method.model
print({'g':base.data.entity_type,'K_g':K_g,'embedding dimension':s.embedding_dim,'epochs':method.optimization.epochs})


In [ ]:
distance=np.linspace(0,4,400)
fig,ax=plt.subplots(figsize=(8,4))
for ell in (.5,1,2): ax.plot(distance,np.exp(-distance**2/(2*ell**2)),label=fr'$\ell={ell}$')
ax.set(xlabel='learned embedding distance $d_{ab}$',ylabel='RBF similarity',title='Kernel restriction induced by the length scale'); ax.legend(); plt.show()


## Estimation on the complete dependence sample

M4 is fitted on every valid complete training vector $\mathbf z_{g,\tau}^{(i)}\in\mathbb R^{K_g}$ and selected by validation pseudo-likelihood, exactly as M2 and M3 are. A case is absent only when at least one member of the fixed group is invalid. There is no method-specific origin truncation or implicit optimization cap. Reduced budgets, when desired for teaching or preliminary work, are explicit overrides in a laboratory composite and have the same meaning for M2, M3, and M4.

In [ ]:
if TRAIN_IF_MISSING and not (RUN_DIR/'best.pt').is_file():
    train_from_config(config,cache_dir=cache_dir,output_dir=RUN_DIR)
if RUN_EVALUATION:
    evaluate_from_config(config,methods=('conditional_kernel',),method_runs={'conditional_kernel':RUN_DIR},cache_dir=cache_dir,output_dir=RUN_DIR/'evaluation')


## Controls and interpretation

`method.model.embedding_dim`, `hidden_dims`, `initial_length_scale`, `nugget`, and `jitter` define the kernel family. `method.optimization.*` defines estimation. `ConditionalKernelGaussianCopula` implements the map. Because the RBF kernel is nonnegative, a poor result may indicate that the learned similarity geometry cannot represent negative conditional association; this is a scientific restriction of M4, not a reduced execution status.